In [1]:
# ============================================================
# MODUL 02: KONTROL MOTOR DC ESP32 VIA PYTHON SERIAL
# Mata Kuliah: Robotika - iMCLab Kit
#
# Deskripsi:
# Program penguji antarmuka serial dua arah (Python - ESP32).
# Memiliki fitur auto-scan port USB-UART, demo sekuensial PWM,
# dan Mode Interaktif Command-Line Interface (CLI).
# ============================================================

import serial
import serial.tools.list_ports
import time
import sys

# ============================================
# KONFIGURASI UTAMA
# ============================================
BAUDRATE = 115200
DUTY_STEP = 10  # Langkah perubahan kecepatan PWM (0-255)

# Catalog ID Chipset USB-UART ESP32 (CP210x, CH340, FTDI)
ESP32_KEYWORDS = [
    "cp210", "cp2102", "cp210x",
    "ch340", "ch341", "ch34x",
    "ftdi", "ft232",
    "usb-serial", "usb serial",
    "usb uart", "uart bridge",
    "silicon labs", "espressif", "esp32",
]

ESP32_VID_PID = {
    (0x10C4, 0xEA60), (0x10C4, 0xEA70),
    (0x1A86, 0x7523), (0x1A86, 0x55D4),
    (0x0403, 0x6001), (0x0403, 0x6015),
    (0x303A, 0x1001),
}


# ============================================
# UTILITAS AUTO-DETECT PORT COM
# ============================================
def list_serial_ports():
    """Menampilkan seluruh daftar port COM yang terhubung ke PC."""
    ports = serial.tools.list_ports.comports()
    if not ports:
        print("[INFO] Tidak ada port serial yang terdeteksi.")
        return []
    print("Daftar Port Serial Aktif:")
    for p in ports:
        vid_pid = f" [VID:PID={p.vid:04X}:{p.pid:04X}]" if (p.vid and p.pid) else ""
        print(f"  - {p.device} : {p.description}{vid_pid}")
    return list(ports)


def deteksi_port_esp32(verbose=True):
    """Mendeteksi port ESP32 berdasarkan skor kecocokan VID/PID & Keyword."""
    ports = serial.tools.list_ports.comports()
    if not ports:
        if verbose:
            print("[WARN] Port serial tidak ditemukan.")
        return None

    kandidat = []
    for p in ports:
        desc = (p.description or "").lower()
        mfg  = (p.manufacturer or "").lower()
        hwid = (p.hwid or "").lower()
        teks = f"{desc} {mfg} {hwid}"

        skor = 0
        if p.vid is not None and p.pid is not None:
            if (p.vid, p.pid) in ESP32_VID_PID:
                skor += 100
        for kw in ESP32_KEYWORDS:
            if kw in teks:
                skor += 50
                break
        if "bluetooth" in teks:
            skor -= 200  # Abaikan port Bluetooth virtual

        if skor > 0:
            kandidat.append((skor, p))

    if not kandidat:
        non_bt = [p for p in ports if "bluetooth" not in (p.description or "").lower()]
        if len(non_bt) == 1:
            if verbose:
                print(f"[INFO] Terdeteksi 1 port fisik non-Bluetooth: {non_bt[0].device}")
            return non_bt[0].device
        if verbose:
            print("[WARN] Port ESP32 gagal teridentifikasi secara otomatis.")
        return None

    kandidat.sort(key=lambda x: x[0], reverse=True)
    if verbose:
        print("\nHasil Pemindaian Port ESP32:")
        for skor, p in kandidat:
            print(f"  [Skor {skor:>3}] {p.device} : {p.description}")
    return kandidat[0][1].device


# ============================================
# KOMUNIKASI SERIAL & KONTROL HARDWARE
# ============================================
def buka_koneksi(port, baudrate=115200):
    """Membuka sesi koneksi serial ke ESP32."""
    try:
        ser = serial.Serial(port, baudrate, timeout=1)
        time.sleep(2)  # Jeda waktu reset booting ESP32
        print(f"[SUCCESS] Terhubung ke ESP32 ({port} @ {baudrate} baud)\n")
        return ser
    except serial.SerialException as e:
        print(f"[ERROR] Gagal membuka port {port}: {e}")
        return None


def kirim_perintah(ser, perintah: str):
    """Mengirim string perintah ke ESP32 dan membaca balasan log."""
    ser.write((perintah + "\n").encode("utf-8"))
    ser.flush()
    time.sleep(0.15)

    while ser.in_waiting > 0:
        balasan = ser.readline().decode("utf-8", errors="ignore").strip()
        if balasan:
            print(f"  [ESP32] -> {balasan}")
        time.sleep(0.05)


def motor_on(ser):
    print(">> Action: Motor ON")
    kirim_perintah(ser, "1")

def motor_off(ser):
    print(">> Action: Motor OFF")
    kirim_perintah(ser, "0")

def kecepatan_naik(ser, step=DUTY_STEP):
    print(f">> Action: Speed +{step}")
    kirim_perintah(ser, "+")

def kecepatan_turun(ser, step=DUTY_STEP):
    print(f">> Action: Speed -{step}")
    kirim_perintah(ser, "-")

def set_pwm(ser, duty: int):
    duty = max(0, min(255, int(duty)))
    print(f">> Action: Set PWM Direct = {duty}")
    kirim_perintah(ser, f"PWM:{duty}")


# ============================================
# PROGRAM UTAMA (MAIN ROUTINE)
# ============================================
if __name__ == "__main__":
    print("=" * 60)
    print("   PEMINDAIAN SISTEM HARDWARE iMCLAB")
    print("=" * 60)
    list_serial_ports()
    print("-" * 60)

    PORT = deteksi_port_esp32()
    if PORT is None:
        print("[FATAL] Koneksi dibatalkan: Port ESP32 tidak ditemukan.")
        sys.exit(1)

    print(f">> Target Port ESP32 Terpilih: {PORT}")
    esp32 = buka_koneksi(PORT, BAUDRATE)

    if esp32 is not None:
        try:
            # --- 1. Uji Coba Sekuensial / Demo Otomatis ---
            print("--- MEMULAI DEMO SEKUENSAL KONTROL MOTOR ---")
            motor_on(esp32)
            time.sleep(2)

            # Iterasi kenaikan PWM (3x)
            for i in range(3):
                kecepatan_naik(esp32)
                time.sleep(1)

            # Iterasi penurunan PWM (2x)
            for i in range(2):
                kecepatan_turun(esp32)
                time.sleep(1)

            # Set nilai PWM spesifik & matikan motor
            set_pwm(esp32, 150)
            time.sleep(2)

            motor_off(esp32)
            print("--- DEMO SEKUENSAL SELESAI ---\n")

            # --- 2. Mode Interaktif CLI ---
            print("=" * 60)
            print("MODE INTERAKTIF KONTROL MOTOR DC:")
            print("  '1'     : Motor ON (Default/Duty Terakhir)")
            print("  '0'     : Motor OFF (Duty = 0)")
            print("  '+'     : Naikkan Kecepatan (+10)")
            print("  '-'     : Turunkan Kecepatan (-10)")
            print("  'p<num>': Set PWM Spesifik (contoh: p120, p200)")
            print("  'q'     : Keluar Program")
            print("=" * 60)

            while True:
                cmd = input("\nPerintah CLI > ").strip()

                if cmd.lower() == "q":
                    print("[INFO] Menghentikan sesi interaktif...")
                    break
                elif cmd == "1":
                    motor_on(esp32)
                elif cmd == "0":
                    motor_off(esp32)
                elif cmd == "+":
                    kecepatan_naik(esp32)
                elif cmd == "-":
                    kecepatan_turun(esp32)
                elif cmd.lower().startswith("p"):
                    try:
                        val = int(cmd[1:])
                        set_pwm(esp32, val)
                    except ValueError:
                        print("[ERR] Format PWM tidak valid! Gunakan format p<angka> (misal: p150).")
                else:
                    print("[ERR] Perintah tidak dikenali. Gunakan (1 / 0 / + / - / p<angka> / q).")

        except KeyboardInterrupt:
            print("\n[WARN] Program dihentikan paksa oleh pengguna.")

        finally:
            try:
                motor_off(esp32)
            except Exception:
                pass
            esp32.close()
            print("[SUCCESS] Koneksi serial berhasil ditutup secara aman.")

   PEMINDAIAN SISTEM HARDWARE iMCLAB
Daftar Port Serial Aktif:
  - COM4 : Silicon Labs CP210x USB to UART Bridge (COM4) [VID:PID=10C4:EA60]
------------------------------------------------------------

Hasil Pemindaian Port ESP32:
  [Skor 150] COM4 : Silicon Labs CP210x USB to UART Bridge (COM4)
>> Target Port ESP32 Terpilih: COM4
[SUCCESS] Terhubung ke ESP32 (COM4 @ 115200 baud)

--- MEMULAI DEMO SEKUENSAL KONTROL MOTOR ---
>> Action: Motor ON
  [ESP32] -> Motor ON, duty = 200
>> Action: Speed +10
  [ESP32] -> Kecepatan NAIK, duty = 210
>> Action: Speed +10
  [ESP32] -> Kecepatan NAIK, duty = 220
>> Action: Speed +10
  [ESP32] -> Kecepatan NAIK, duty = 230
>> Action: Speed -10
  [ESP32] -> Kecepatan TURUN, duty = 220
>> Action: Speed -10
  [ESP32] -> Kecepatan TURUN, duty = 210
>> Action: Set PWM Direct = 150
  [ESP32] -> Set duty = 150
>> Action: Motor OFF
  [ESP32] -> Motor OFF, duty = 0
--- DEMO SEKUENSAL SELESAI ---

MODE INTERAKTIF KONTROL MOTOR DC:
  '1'     : Motor ON (Default/